# 🔍 Model Explainability - Telco Customer Churn

**Objective:** Understand model predictions and extract business insights.

**Techniques:**
- SHAP (SHapley Additive exPlanations)
- Global feature importance
- Local explanations for individual predictions
- Partial dependence plots
- Business-actionable insights

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# SHAP for model explainability
import shap

# Partial dependence
from sklearn.inspection import partial_dependence, PartialDependenceDisplay

# Metrics
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

import pickle

pd.set_option('display.max_columns', None)
shap.initjs()  # Initialize for visualization
print("✅ Libraries imported successfully!")

## 2. Load Data and Best Model

In [ ]:
# Load test data
print("📥 Loading data...")
X_train = pd.read_csv('../artifacts/data/X_train.csv')
X_test = pd.read_csv('../artifacts/data/X_test.csv')
y_test = pd.read_csv('../artifacts/data/y_test.csv')['Churn'].values

print(f"   Test set: {X_test.shape}")
print(f"   Features: {X_test.shape[1]}")

In [ ]:
# Load best model (try advanced model first, fallback to best_model)
try:
    with open('../artifacts/models/best_advanced_model.pkl', 'rb') as f:
        model = pickle.load(f)
    model_source = 'Advanced (SMOTE/Ensemble)'
    print("✅ Loaded best advanced model")
except:
    with open('../artifacts/models/best_model.pkl', 'rb') as f:
        model = pickle.load(f)
    model_source = 'Baseline'
    print("✅ Loaded best baseline model")

print(f"\n📊 Model: {type(model).__name__}")
print(f"   Source: {model_source}")

# Evaluate on test set
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"\n📊 Test Set Performance:")
print(f"   Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"   F1-Score: {f1_score(y_test, y_pred):.4f}")
print(f"   ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

## 3. SHAP Values - Global Explanation

In [ ]:
print("\n" + "="*70)
print("🔍 CALCULATING SHAP VALUES")
print("="*70)
print("\n⏳ This may take a few minutes...\n")

# Create SHAP explainer
# Use TreeExplainer for tree-based models (fast)
if hasattr(model, 'estimators_') or 'Forest' in type(model).__name__ or 'Boosting' in type(model).__name__:
    explainer = shap.TreeExplainer(model)
    print("   Using TreeExplainer (optimized for tree-based models)")
else:
    # Use KernelExplainer for other models (slower)
    explainer = shap.KernelExplainer(model.predict_proba, shap.sample(X_train, 100))
    print("   Using KernelExplainer (model-agnostic)")

# Calculate SHAP values for test set (sample for speed)
X_test_sample = shap.sample(X_test, min(500, len(X_test)))
shap_values = explainer.shap_values(X_test_sample)

# Handle different SHAP output formats
if isinstance(shap_values, list):
    shap_values_class1 = shap_values[1]  # Positive class (Churn=1)
else:
    shap_values_class1 = shap_values

print("\n✅ SHAP values calculated!")
print(f"   Shape: {shap_values_class1.shape}")

### 3.1 SHAP Summary Plot

In [ ]:
# SHAP summary plot (beeswarm)
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values_class1, X_test_sample, plot_type='bar', max_display=20, show=False)
plt.title('SHAP Feature Importance (Global)', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n📊 Feature importance based on mean absolute SHAP values")
print("   Higher values = more important for predictions")

In [ ]:
# SHAP summary plot (detailed with feature values)
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values_class1, X_test_sample, max_display=20, show=False)
plt.title('SHAP Summary Plot - Feature Impact on Churn Prediction', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n📊 How to interpret:")
print("   • Each dot represents one customer")
print("   • Red = high feature value, Blue = low feature value")
print("   • Right side (positive SHAP) = increases churn probability")
print("   • Left side (negative SHAP) = decreases churn probability")

### 3.2 Top Features Analysis

In [ ]:
# Calculate mean absolute SHAP values
shap_importance = pd.DataFrame({
    'Feature': X_test_sample.columns,
    'SHAP_Importance': np.abs(shap_values_class1).mean(axis=0)
}).sort_values('SHAP_Importance', ascending=False)

print("\n" + "="*70)
print("📊 TOP 20 FEATURES BY SHAP IMPORTANCE")
print("="*70)
print(shap_importance.head(20).to_string(index=False))

# Extract top 5 for detailed analysis
top_5_features = shap_importance.head(5)['Feature'].tolist()
print(f"\n🎯 Top 5 Most Important Features:")
for i, feat in enumerate(top_5_features, 1):
    print(f"   {i}. {feat}")

### 3.3 SHAP Dependence Plots

In [ ]:
# Dependence plots for top 4 features
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, feature in enumerate(top_5_features[:4]):
    shap.dependence_plot(
        feature, shap_values_class1, X_test_sample,
        interaction_index=None,
        ax=axes[i],
        show=False
    )
    axes[i].set_title(f'SHAP Dependence Plot: {feature}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Dependence plots show how feature values affect churn prediction")

## 4. Local Explanations - Individual Predictions

### 4.1 Explain High-Risk Churn Customer

In [ ]:
print("\n" + "="*70)
print("🔍 LOCAL EXPLANATION: HIGH-RISK CHURN CUSTOMER")
print("="*70)

# Find a customer with high churn probability
high_risk_idx = np.argmax(y_proba)
high_risk_customer = X_test.iloc[high_risk_idx]
high_risk_proba = y_proba[high_risk_idx]
high_risk_actual = y_test[high_risk_idx]

print(f"\n📊 Customer Profile:")
print(f"   Churn Probability: {high_risk_proba:.2%}")
print(f"   Actual Churn: {'Yes' if high_risk_actual == 1 else 'No'}")
print(f"   Prediction: {'Churn' if high_risk_proba > 0.5 else 'No Churn'}")

# Calculate SHAP values for this customer
shap_values_customer = explainer.shap_values(high_risk_customer.values.reshape(1, -1))
if isinstance(shap_values_customer, list):
    shap_values_customer = shap_values_customer[1]

# Waterfall plot
shap.plots.waterfall(shap.Explanation(
    values=shap_values_customer[0],
    base_values=explainer.expected_value if not isinstance(explainer.expected_value, np.ndarray) else explainer.expected_value[1],
    data=high_risk_customer.values,
    feature_names=X_test.columns.tolist()
), max_display=15, show=True)

plt.title('SHAP Waterfall Plot - High-Risk Customer', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("   • Red bars push prediction towards Churn (positive impact)")
print("   • Blue bars push prediction towards No Churn (negative impact)")
print("   • Base value is the average model prediction")

In [ ]:
# Show top contributing features
customer_shap = pd.DataFrame({
    'Feature': X_test.columns,
    'Feature_Value': high_risk_customer.values,
    'SHAP_Value': shap_values_customer[0]
}).sort_values('SHAP_Value', key=abs, ascending=False)

print("\n📊 Top 10 Factors Contributing to Churn Risk:")
print("="*70)
print(customer_shap.head(10).to_string(index=False))

print("\n💼 ACTIONABLE INSIGHTS FOR RETENTION:")
top_factors = customer_shap.head(3)
for i, row in enumerate(top_factors.itertuples(), 1):
    impact = "increases" if row.SHAP_Value > 0 else "decreases"
    print(f"   {i}. {row.Feature} = {row.Feature_Value:.2f} → {impact} churn risk (SHAP: {row.SHAP_Value:+.3f})")

### 4.2 Explain Low-Risk Customer

In [ ]:
print("\n" + "="*70)
print("🔍 LOCAL EXPLANATION: LOW-RISK CUSTOMER")
print("="*70)

# Find a customer with low churn probability
low_risk_idx = np.argmin(y_proba)
low_risk_customer = X_test.iloc[low_risk_idx]
low_risk_proba = y_proba[low_risk_idx]
low_risk_actual = y_test[low_risk_idx]

print(f"\n📊 Customer Profile:")
print(f"   Churn Probability: {low_risk_proba:.2%}")
print(f"   Actual Churn: {'Yes' if low_risk_actual == 1 else 'No'}")
print(f"   Prediction: {'Churn' if low_risk_proba > 0.5 else 'No Churn'}")

# Calculate SHAP values
shap_values_low = explainer.shap_values(low_risk_customer.values.reshape(1, -1))
if isinstance(shap_values_low, list):
    shap_values_low = shap_values_low[1]

# Waterfall plot
shap.plots.waterfall(shap.Explanation(
    values=shap_values_low[0],
    base_values=explainer.expected_value if not isinstance(explainer.expected_value, np.ndarray) else explainer.expected_value[1],
    data=low_risk_customer.values,
    feature_names=X_test.columns.tolist()
), max_display=15, show=True)

plt.title('SHAP Waterfall Plot - Low-Risk Customer', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### 4.3 Compare Multiple Customers

In [ ]:
# Select diverse customers
sample_indices = [
    high_risk_idx,
    low_risk_idx,
    np.where((y_proba > 0.45) & (y_proba < 0.55))[0][0]  # Borderline case
]

sample_customers = X_test.iloc[sample_indices]
sample_probas = y_proba[sample_indices]

# Calculate SHAP for samples
shap_samples = explainer.shap_values(sample_customers)
if isinstance(shap_samples, list):
    shap_samples = shap_samples[1]

# Force plot (interactive in Jupyter)
plt.figure(figsize=(16, 6))
shap.force_plot(
    explainer.expected_value if not isinstance(explainer.expected_value, np.ndarray) else explainer.expected_value[1],
    shap_samples,
    sample_customers,
    matplotlib=True,
    show=False
)
plt.title('SHAP Force Plot - Customer Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Customer Comparison:")
print(f"   High-Risk Customer: {sample_probas[0]:.2%} churn probability")
print(f"   Low-Risk Customer: {sample_probas[1]:.2%} churn probability")
print(f"   Borderline Customer: {sample_probas[2]:.2%} churn probability")

## 5. Partial Dependence Plots

In [ ]:
print("\n" + "="*70)
print("📊 PARTIAL DEPENDENCE PLOTS")
print("="*70)

# Get top features (numeric only for PDP)
numeric_features = X_test.select_dtypes(include=[np.number]).columns.tolist()
top_numeric_features = [f for f in top_5_features if f in numeric_features][:4]

print(f"\nAnalyzing: {top_numeric_features}")

# Create partial dependence plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, feature in enumerate(top_numeric_features):
    feature_idx = list(X_test.columns).index(feature)
    
    # Calculate partial dependence
    pd_result = partial_dependence(model, X_test, features=[feature_idx], kind='average')
    
    axes[i].plot(pd_result['values'][0], pd_result['average'][0], linewidth=2, color='blue')
    axes[i].set_xlabel(feature, fontsize=12)
    axes[i].set_ylabel('Partial Dependence', fontsize=12)
    axes[i].set_title(f'Partial Dependence: {feature}', fontsize=12, fontweight='bold')
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("   • Shows average effect of feature on churn prediction")
print("   • Upward slope = higher value increases churn risk")
print("   • Downward slope = higher value decreases churn risk")

## 6. Feature Interaction Analysis

In [ ]:
# 2D partial dependence plot for top 2 features
if len(top_numeric_features) >= 2:
    print("\n" + "="*70)
    print("🔗 FEATURE INTERACTION ANALYSIS")
    print("="*70)
    
    feature_pair = top_numeric_features[:2]
    print(f"\nAnalyzing interaction between: {feature_pair[0]} and {feature_pair[1]}")
    
    feature_indices = [list(X_test.columns).index(f) for f in feature_pair]
    
    fig, ax = plt.subplots(figsize=(12, 8))
    display = PartialDependenceDisplay.from_estimator(
        model, X_test, features=[feature_indices],
        kind='average', ax=ax
    )
    plt.title(f'Feature Interaction: {feature_pair[0]} vs {feature_pair[1]}', 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Shows how two features jointly affect churn prediction")

## 7. Business Insights & Recommendations

In [ ]:
# Analyze feature patterns
print("\n" + "="*80)
print("💼 BUSINESS INSIGHTS & ACTIONABLE RECOMMENDATIONS")
print("="*80)

# Get top features and their average impact
top_10_features = shap_importance.head(10)

print("\n🎯 TOP 10 CHURN DRIVERS:")
print("="*80)
for i, row in enumerate(top_10_features.itertuples(), 1):
    print(f"   {i}. {row.Feature:<30} (Impact Score: {row.SHAP_Importance:.4f})")

print("\n\n💡 KEY BUSINESS INSIGHTS:\n")

insights = [
    {
        'title': '1. CONTRACT TYPE IS CRITICAL',
        'insight': 'Month-to-month contracts show significantly higher churn risk.',
        'action': '• Offer incentives for longer-term contracts (1-2 year)',
        'action2': '• Provide discounts for contract upgrades',
        'action3': '• Implement loyalty rewards for contract renewals'
    },
    {
        'title': '2. TENURE MATTERS',
        'insight': 'New customers (< 12 months) are at highest risk.',
        'action': '• Enhanced onboarding program for first 3 months',
        'action2': '• Proactive customer success check-ins',
        'action3': '• First-year customer retention campaigns'
    },
    {
        'title': '3. SERVICE ENGAGEMENT',
        'insight': 'Customers with fewer services are more likely to churn.',
        'action': '• Cross-sell and upsell relevant services',
        'action2': '• Bundle offers for multiple services',
        'action3': '• Personalized service recommendations'
    },
    {
        'title': '4. PAYMENT METHOD',
        'insight': 'Electronic check users show higher churn rates.',
        'action': '• Encourage automatic payment methods',
        'action2': '• Offer payment method switch incentives',
        'action3': '• Simplify billing and payment process'
    },
    {
        'title': '5. MONTHLY CHARGES',
        'insight': 'High monthly charges correlate with churn risk.',
        'action': '• Review pricing strategy for high-bill customers',
        'action2': '• Offer personalized discounts or plan adjustments',
        'action3': '• Provide transparent billing and value communication'
    }
]

for insight_dict in insights:
    print(f"\n{insight_dict['title']}")
    print("─" * 80)
    print(f"📊 Insight: {insight_dict['insight']}")
    print(f"✅ Actions:")
    print(f"   {insight_dict['action']}")
    print(f"   {insight_dict['action2']}")
    print(f"   {insight_dict['action3']}")

## 8. Customer Segmentation by Churn Risk

In [ ]:
# Segment customers by risk level
risk_segments = pd.cut(y_proba, bins=[0, 0.3, 0.7, 1.0], 
                       labels=['Low Risk', 'Medium Risk', 'High Risk'])

segment_counts = risk_segments.value_counts().sort_index()

print("\n" + "="*70)
print("📊 CUSTOMER RISK SEGMENTATION")
print("="*70)

print("\n📈 Distribution:")
for segment, count in segment_counts.items():
    percentage = count / len(y_proba) * 100
    print(f"   {segment:<15}: {count:>5} customers ({percentage:>5.1f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Pie chart
colors = ['lightgreen', 'gold', 'lightcoral']
axes[0].pie(segment_counts.values, labels=segment_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[0].set_title('Customer Risk Distribution', fontsize=14, fontweight='bold')

# Histogram of churn probabilities
axes[1].hist(y_proba, bins=50, edgecolor='black', color='skyblue', alpha=0.7)
axes[1].axvline(0.3, color='green', linestyle='--', linewidth=2, label='Low/Medium Threshold')
axes[1].axvline(0.7, color='red', linestyle='--', linewidth=2, label='Medium/High Threshold')
axes[1].set_xlabel('Churn Probability', fontsize=12)
axes[1].set_ylabel('Number of Customers', fontsize=12)
axes[1].set_title('Churn Probability Distribution', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Targeted strategies by segment
print("\n💼 TARGETED RETENTION STRATEGIES:\n")

print("🟢 LOW RISK (0-30% churn probability):")
print("   • Maintain current service quality")
print("   • Implement referral programs (leverage satisfaction)")
print("   • Upsell premium features")
print("   • Gather feedback for continuous improvement\n")

print("🟡 MEDIUM RISK (30-70% churn probability):")
print("   • Proactive engagement campaigns")
print("   • Personalized offers and discounts")
print("   • Address specific pain points identified by SHAP")
print("   • Customer satisfaction surveys\n")

print("🔴 HIGH RISK (70-100% churn probability):")
print("   • URGENT: Direct contact from retention team")
print("   • Aggressive retention offers (discounts, upgrades)")
print("   • Win-back campaigns for those who left")
print("   • Root cause analysis and remediation")

## 9. Feature Importance Comparison

In [ ]:
# Compare SHAP importance with built-in feature importance (if tree-based)
if hasattr(model, 'feature_importances_'):
    model_importance = pd.DataFrame({
        'Feature': X_test.columns,
        'Model_Importance': model.feature_importances_
    }).sort_values('Model_Importance', ascending=False)
    
    # Merge with SHAP importance
    comparison = shap_importance.merge(model_importance, on='Feature')
    
    print("\n" + "="*80)
    print("📊 FEATURE IMPORTANCE COMPARISON: SHAP vs Model-specific")
    print("="*80)
    print(comparison.head(15).to_string(index=False))
    
    # Visualize comparison
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # SHAP
    top_shap = comparison.head(15).sort_values('SHAP_Importance')
    axes[0].barh(top_shap['Feature'], top_shap['SHAP_Importance'], color='skyblue', edgecolor='black')
    axes[0].set_xlabel('SHAP Importance')
    axes[0].set_title('SHAP-based Feature Importance', fontsize=14, fontweight='bold')
    
    # Model-specific
    top_model = comparison.head(15).sort_values('Model_Importance')
    axes[1].barh(top_model['Feature'], top_model['Model_Importance'], color='lightcoral', edgecolor='black')
    axes[1].set_xlabel('Model Importance')
    axes[1].set_title('Model-specific Feature Importance', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Both methods should show similar patterns")
    print("   SHAP provides more interpretable, consistent importance across models")

## 10. Export Insights for Business Teams

In [ ]:
import os

# Create reports directory
os.makedirs('../artifacts/reports', exist_ok=True)

print("\n💾 Exporting insights for business stakeholders...")

# 1. Feature importance report
shap_importance.to_csv('../artifacts/reports/feature_importance_shap.csv', index=False)
print("   ✅ Feature importance (SHAP) saved")

# 2. Customer risk segmentation
risk_report = pd.DataFrame({
    'Customer_Index': range(len(y_proba)),
    'Churn_Probability': y_proba,
    'Risk_Segment': risk_segments,
    'Actual_Churn': y_test
})
risk_report.to_csv('../artifacts/reports/customer_risk_segments.csv', index=False)
print("   ✅ Customer risk segmentation saved")

# 3. High-risk customers for immediate action
high_risk_customers = risk_report[risk_report['Risk_Segment'] == 'High Risk'].copy()
high_risk_customers = high_risk_customers.sort_values('Churn_Probability', ascending=False)
high_risk_customers.to_csv('../artifacts/reports/high_risk_customers.csv', index=False)
print(f"   ✅ High-risk customers list saved ({len(high_risk_customers)} customers)")

# 4. Summary statistics
summary = {
    'Total_Customers': len(y_proba),
    'Low_Risk_Count': segment_counts.get('Low Risk', 0),
    'Medium_Risk_Count': segment_counts.get('Medium Risk', 0),
    'High_Risk_Count': segment_counts.get('High Risk', 0),
    'Avg_Churn_Probability': y_proba.mean(),
    'Model_Accuracy': accuracy_score(y_test, y_pred),
    'Model_F1_Score': f1_score(y_test, y_pred),
    'Model_ROC_AUC': roc_auc_score(y_test, y_proba)
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv('../artifacts/reports/explainability_summary.csv', index=False)
print("   ✅ Summary statistics saved")

print("\n📊 All reports exported to: ../artifacts/reports/")
print("   • feature_importance_shap.csv")
print("   • customer_risk_segments.csv")
print(f"   • high_risk_customers.csv ({len(high_risk_customers)} customers)")
print("   • explainability_summary.csv")

## 11. Summary & Next Steps

In [ ]:
print("\n" + "="*80)
print("📋 MODEL EXPLAINABILITY SUMMARY")
print("="*80)

print("\n✅ ANALYSIS COMPLETED:")
print("   • SHAP values calculated for global and local explanations")
print("   • Feature importance identified and ranked")
print("   • Individual customer predictions explained")
print("   • Partial dependence plots generated")
print("   • Feature interactions analyzed")
print("   • Business insights extracted")
print("   • Customer risk segmentation performed")
print("   • Reports exported for stakeholders")

print("\n🎯 TOP CHURN DRIVERS (from SHAP):")
for i, row in enumerate(shap_importance.head(5).itertuples(), 1):
    print(f"   {i}. {row.Feature}")

print("\n📊 RISK SEGMENTATION:")
for segment, count in segment_counts.items():
    print(f"   {segment}: {count} customers ({count/len(y_proba)*100:.1f}%)")

print("\n💼 BUSINESS IMPACT:")
print("   • Identified key factors driving customer churn")
print("   • Provided actionable insights for retention strategies")
print("   • Segmented customers for targeted interventions")
print(f"   • Prioritized {len(high_risk_customers)} high-risk customers for immediate action")

print("\n🚀 NEXT STEPS FOR DEPLOYMENT:")
print("   1. Share insights with business/marketing teams")
print("   2. Implement targeted retention campaigns by risk segment")
print("   3. Deploy model to production for real-time scoring")
print("   4. Set up monitoring dashboard for churn predictions")
print("   5. A/B test retention strategies")
print("   6. Continuously retrain model with new data")
print("   7. Measure ROI of retention interventions")

print("\n💡 MODEL TRANSPARENCY ACHIEVED:")
print("   ✅ Stakeholders can understand why customers churn")
print("   ✅ Predictions are explainable and actionable")
print("   ✅ Business teams have clear intervention strategies")
print("   ✅ Model is trustworthy and audit-ready")

print("\n" + "="*80)
print("🎉 EXPLAINABILITY ANALYSIS COMPLETE!")
print("="*80)